In [ ]:
import pandas as pd

In [ ]:
path_1 = 'molecule_results_ranked.csv'
path_2 = 'nmdn_output.csv'
path_3 = 'gnina_output.csv'
diffdock_output = pd.read_csv(path_1)
nmdn_output = pd.read_csv(path_2)
gnina_output = pd.read_csv(path_3)
nmdn_output.rename(columns={'lig': 'ligand'}, inplace=True)
actives_smiles_path = 'actives.smi'
diffdock_output.head()

In [ ]:
nmdn_output

In [ ]:
def clean_name_nmdn(df):
    for index, row in df.iterrows():
        # Extract the file name from the 'ligand' column
        part = row['ligand'].split('/')[-1]
        # Remove the '_best_pose.sdf' suffix
        part = part.replace('_best_pose.sdf', '')
        # Update the column 'ligand' to 'drug'
        df.at[index, 'drug'] = str(part)  # Ensure string type
    # Ensure the entire column is string type
    df['drug'] = df['drug'].astype(str)
    return df

nmdn_output = clean_name_nmdn(nmdn_output)
nmdn_output['drug'] = nmdn_output['drug'].astype(str)

nmdn_output.sort_values('pKd-Score', ascending=False).head()


In [ ]:
gnina_output['drug'] = gnina_output['drug'].astype(str)
gnina_output

In [ ]:
diffdock_output.rename(columns={'molecule_name': 'drug'}, inplace=True)
diffdock_output['drug'] = diffdock_output['drug'].astype(str)

# Labeling Actives

In [ ]:
def get_actives_names(actives_smiles_path):
    actives = []
    with open(actives_smiles_path, 'r') as f:
        for line in f:
            line = line.strip()  # Remove whitespace
            if line:  # Skip empty lines
                parts = line.split()
                if len(parts) >= 2:  # Ensure we have at least 2 parts
                    actives.append(parts[1])
                else:
                    print(f"Warning: Skipping malformed line: {line}")
    return actives

actives = get_actives_names(actives_smiles_path)
print(f"Total active compounds: {len(actives)}")
print(f"Sample actives: {actives[:10]}")

def label_actives(df, actives):
    # Convert actives to set for faster lookup
    actives_set = set(actives)
    
    # Vectorized approach - much faster
    df['Active'] = df['drug'].astype(str).isin(actives_set)
    
    return df

# Apply to all datasets
diffdock_output = label_actives(diffdock_output, actives)
nmdn_output = label_actives(nmdn_output, actives)
gnina_output = label_actives(gnina_output, actives)

print(f"\nActive labeling completed:")
print(f"AutoDock actives: {diffdock_output['Active'].sum()}/{len(diffdock_output)}")
print(f"NMDN actives: {nmdn_output['Active'].sum()}/{len(nmdn_output)}")
print(f"GNINA actives: {gnina_output['Active'].sum()}/{len(gnina_output)}")


In [ ]:
gnina_output.head(50)

# filtering

In [ ]:
total_compounds = len(nmdn_output)
top_10_percent = int(len(nmdn_output) * 0.1)
top_1_percent = int(len(nmdn_output) * 0.01)
total_actives = nmdn_output['Active'].sum()
def calculate_ef10(df):
    # Sort by docking score (assuming lower scores are better)

    # Calculate 10% of total compounds
    

    # Get top 10% compounds
    top_10_percent_compounds = df.head(top_10_percent)

    # Count actives in top 10%
    actives_in_top_10_percent = top_10_percent_compounds['Active'].sum()

    # Total number of actives in dataset
    # total_actives = df['Active'].sum()

    # Calculate EF10%
    if total_actives == 0:
        ef10 = 0
    else:
        ef10 = (actives_in_top_10_percent / top_10_percent) / (total_actives / total_compounds)
    return ef10

def calculate_ef1(df):
    # Sort by docking score (assuming lower scores are better)

    # Calculate 10% of total compounds
    

    # Get top 1% compounds
    top_1_percent_compounds = df.head(top_1_percent)

    # Count actives in top 1%
    actives_in_top_1_percent = top_1_percent_compounds['Active'].sum()

    # Total number of actives in dataset
    # total_actives = df['Active'].sum()

    # Calculate EF10%
    if total_actives == 0:
        ef1 = 0
    else:
        ef1 = (actives_in_top_1_percent / top_1_percent) / (total_actives / total_compounds)
    return ef1

def calculate_ef(df):
    # Sort by docking score (assuming lower scores are better)

    # Calculate 10% of total compounds
    top_10_percent = int(len(df))
    if top_10_percent == 0:
        return 0

    # Get top 10% compounds
    top_10_percent_compounds = df.head(top_10_percent)

    # Count actives in top 10%
    actives_in_top_10_percent = top_10_percent_compounds['Active'].sum()

    # Total number of actives in dataset
    # total_actives = df['Active'].sum()

    # Calculate EF10%
    if total_actives == 0:
        return 0
    else:
        ef10 = (actives_in_top_10_percent / top_10_percent) / (total_actives / total_compounds)
    return ef10



In [ ]:
print (f"\nTotal compounds: {total_compounds}")
print (f"Top 10% compounds: {top_10_percent}")
print (f"Top 1% compounds: {top_1_percent}")
print (f"Total actives: {total_actives}")

ef_10_autodock = calculate_ef10(diffdock_output)
print(f"\nAutoDock EF10: {ef_10_autodock:.6f}")

top_10_percent_autodock = diffdock_output.head(top_10_percent)
top_1_percent_autodock = diffdock_output.head(top_1_percent)
print(f"Top 10% actives: {top_10_percent_autodock['Active'].sum()}")
print(f"Top 1% actives: {top_1_percent_autodock['Active'].sum()}")

ef_1_autodock = calculate_ef1(diffdock_output)
print(f"AutoDock EF1: {ef_1_autodock:.6f}")

ef_autodock = calculate_ef(diffdock_output)
print(f"AutoDock EF: {ef_autodock:.6f}")

# Sort datasets by their primary scoring variables for proper ranking
nmdn_sorted = nmdn_output.sort_values("pKd-Score", ascending=False).reset_index(drop=True)
gnina_sorted = gnina_output.sort_values("CNNaffinity", ascending=False).reset_index(drop=True)

ef_10_nmdn = calculate_ef10(nmdn_sorted)
print(f"\nNMDN EF10: {ef_10_nmdn:.6f}")
top_10_percent_nmdn = nmdn_sorted.head(top_10_percent)
top_1_percent_nmdn = nmdn_sorted.head(top_1_percent)
print(f"Top 10% actives: {top_10_percent_nmdn['Active'].sum()}")
print(f"Top 1% actives: {top_1_percent_nmdn['Active'].sum()}")

ef_1_nmdn = calculate_ef1(nmdn_sorted)
print(f"NMDN EF1: {ef_1_nmdn:.6f}")

ef_nmdn = calculate_ef(nmdn_sorted)
print(f"NMDN EF: {ef_nmdn:.6f}")

ef_10_gnina = calculate_ef10(gnina_sorted)
print(f"\nGNINA EF10: {ef_10_gnina:.6f}")
top_10_percent_gnina = gnina_sorted.head(top_10_percent)
top_1_percent_gnina = gnina_sorted.head(top_1_percent)
print(f"Top 10% actives: {top_10_percent_gnina['Active'].sum()}")
print(f"Top 1% actives: {top_1_percent_gnina['Active'].sum()}")

ef_1_gnina = calculate_ef1(gnina_sorted)
print(f"GNINA EF1: {ef_1_gnina:.6f}")

ef_gnina = calculate_ef(gnina_sorted)
print(f"GNINA EF: {ef_gnina:.6f}")

In [ ]:
# Clean the confidence_score column before merging
print("Cleaning confidence_score column...")
print(f"Original data types in confidence_score: {diffdock_output['confidence_score'].apply(type).value_counts()}")

# Convert confidence_score to numeric, coercing errors to NaN
diffdock_output['confidence_score'] = pd.to_numeric(diffdock_output['confidence_score'], errors='coerce')

# Check for NaN values
nan_count = diffdock_output['confidence_score'].isna().sum()
print(f"Number of NaN values after conversion: {nan_count}")

# Remove rows where confidence_score is NaN
if nan_count > 0:
    print(f"Removing {nan_count} rows with invalid confidence_score values...")
    diffdock_output = diffdock_output.dropna(subset=['confidence_score'])

print(f"After cleaning - Shape: {diffdock_output.shape}")
print(f"Data type: {diffdock_output['confidence_score'].dtype}")



In [ ]:
def merge_and_rank(nmdn_output, gnina_output, diffdock_output, NMDN_Score=-1000, NMDN_weight=1, gnina_weight=1, CNNscore=0, autodock_weight=1):
    """ Merge NMDN and GNINA outputs, rank by combined scores, and filter based on NMDN score threshold.
    """ 
    # Filter NMDN output based on the NMDN_Score threshold
    nmdn_filtered = nmdn_output[nmdn_output['NMDN-Score'] >= NMDN_Score].copy()
    gnina_filtered = gnina_output[gnina_output['CNNscore'] >= CNNscore].copy()

    # Sort them by their primary scoring variables (higher is better)
    nmdn_filtered = nmdn_filtered.sort_values('pKd-Score', ascending=False).reset_index(drop=True)
    gnina_filtered = gnina_filtered.sort_values('CNNaffinity', ascending=False).reset_index(drop=True)
    autodock_sorted = diffdock_output.sort_values('confidence_score', ascending=False).reset_index(drop=True)

    # Add ranks
    nmdn_filtered['rank_nmdn'] = nmdn_filtered.index + 1
    gnina_filtered['rank_gnina'] = gnina_filtered.index + 1
    autodock_sorted['rank_autodock'] = autodock_sorted.index + 1
    
    # Merge datasets - inner join to get mutual drugs only
    merged = pd.merge(nmdn_filtered, gnina_filtered, on='drug', suffixes=('_nmdn', '_gnina'), how='inner')

    # Merge with diffdock_output to include autodock scores for the mutual drugs
    merged = pd.merge(merged, autodock_sorted[['drug', 'confidence_score', 'rank_autodock']], on='drug', how='left')

    # Handle missing AutoDock ranks (assign worst rank + 1)
    max_autodock_rank = autodock_sorted['rank_autodock'].max()
    merged['rank_autodock'] = merged['rank_autodock'].fillna(max_autodock_rank + 1)

    # Calculate weighted mean rank (lower is better)
    rank_total = NMDN_weight + gnina_weight + autodock_weight
    merged['mean_rank'] = (merged['rank_nmdn'] * NMDN_weight + merged['rank_gnina'] * gnina_weight + merged['rank_autodock'] * autodock_weight) / rank_total
    
    # Sort by mean_rank (ascending - lower is better)
    merged = merged.sort_values('mean_rank').reset_index(drop=True)
    merged = merged.rename(columns={'Active_nmdn': 'Active'})

    return merged

# Test the function with the corrected logic
merged = merge_and_rank(nmdn_output, gnina_output, diffdock_output, NMDN_Score=-800, NMDN_weight=1, gnina_weight=2, CNNscore=0.1, autodock_weight=1)
tiny_merged = merge_and_rank(nmdn_output, gnina_output, diffdock_output, NMDN_Score=900, NMDN_weight=1, gnina_weight=1, CNNscore=0.6, autodock_weight=1)
tiny_merged_weighted = merge_and_rank(nmdn_output, gnina_output, diffdock_output, NMDN_Score=-4000, NMDN_weight=1, gnina_weight=2, CNNscore=0, autodock_weight=1)


In [ ]:
for_merge = merge_and_rank(nmdn_output, gnina_output, diffdock_output, NMDN_Score=-100000, NMDN_weight=1, gnina_weight=2, CNNscore=0, autodock_weight=1)
for_merge = for_merge.rename(columns={'rank_autodock': 'rank_diffdock'})
for_merge

In [ ]:
for_merge_autodock = pd.read_csv('for_merge_autodock.csv')

# Convert drug columns to string in both DataFrames
for_merge['drug'] = for_merge['drug'].astype(str)
for_merge_autodock['drug'] = for_merge_autodock['drug'].astype(str)

def merge_two_sets (diffdock, autodock, target_name):
    merged = pd.merge(diffdock, autodock, on='drug', suffixes=('_diffdock', '_autodock'), how='inner')
    merged['target'] = target_name
    return merged

merged_globally = merge_two_sets(for_merge, for_merge_autodock, 'ADRB2')
merged_globally.to_csv('merged_ADRB2.csv', index=False)
merged_globally

In [ ]:
def rank_globally(merged_globally=merged_globally, NMDN_Score=-1000, NMDN_weight=1, gnina_weight=1, CNNscore=0, autodock_weight=1):
    """ Merge NMDN and GNINA outputs, rank by combined scores, and filter based on NMDN score threshold.
    """ 
    # Filter NMDN output based on the NMDN_Score threshold
    merged_globally = merged_globally[merged_globally['NMDN-Score_diffdock'] >= NMDN_Score]
    merged_globally = merged_globally[merged_globally['CNNscore_diffdock'] >= CNNscore]

    

    # Calculate weighted mean rank (lower is better)
    rank_total = NMDN_weight + gnina_weight + autodock_weight
    rank_total = rank_total * 2
    merged_globally['mean_rank'] = (merged_globally['rank_nmdn_diffdock'] * NMDN_weight + merged_globally['rank_gnina_diffdock'] * gnina_weight + merged_globally['rank_diffdock'] * autodock_weight + merged_globally['rank_nmdn_autodock'] * NMDN_weight + merged_globally['rank_gnina_autodock'] * gnina_weight + merged_globally['rank_autodock'] * autodock_weight) / rank_total

    # Sort by mean_rank (ascending - lower is better)
    merged_globally = merged_globally.sort_values('mean_rank').reset_index(drop=True)
    merged_globally = merged_globally.rename(columns={'Active_autodock': 'Active'})

    return merged_globally

In [ ]:
merged_globally = rank_globally(merged_globally=merged_globally, NMDN_Score=-800, NMDN_weight=1, gnina_weight=2, CNNscore=0.1, autodock_weight=1)
merged_globally

In [ ]:
# Calculate EF metrics for merged dataset
merged_ef_10 = calculate_ef10(merged)
top_10_percent_size = int(len(merged) * 0.1)
top_1_percent_size = int(len(merged) * 0.01)
merged_ef_1 = calculate_ef1(merged)
merged_ef = calculate_ef(merged)

# Count actives in merged dataset
merged_actives = merged['Active'].sum()

# Count actives in top 10% and top 1%
top_10_percent_actives = merged.head(top_10_percent_size)['Active'].sum()
top_1_percent_actives = merged.head(top_1_percent_size)['Active'].sum()

print(f"\nTotal compounds in merged dataset: {len(merged)}")
print(f"Number of actives in merged dataset: {merged_actives}")

print(f"\nTop 10% size: {top_10_percent_size}")
print(f"Number of actives in top 10%: {top_10_percent_actives}")
print(f"Merged EF10: {merged_ef_10:.6f}")

print(f"\nTop 1% size: {top_1_percent_size}")
print(f"Number of actives in top 1%: {top_1_percent_actives}")
print(f"Merged EF1: {merged_ef_1:.6f}")

print("=" * 50)

# Calculate EF metrics for tiny_merged dataset
tiny_merged_ef = calculate_ef(tiny_merged)
tiny_merged_size = len(tiny_merged)
tiny_merged_actives = tiny_merged['Active'].sum()

# Count actives in top 10% and top 1% for tiny_merged
tiny_top_10_percent_size = int(tiny_merged_size * 0.1)
tiny_top_1_percent_size = int(tiny_merged_size * 0.01)
tiny_top_10_percent_actives = tiny_merged.head(tiny_top_10_percent_size)['Active'].sum()
tiny_top_1_percent_actives = tiny_merged.head(tiny_top_1_percent_size)['Active'].sum()

print(f"\nTiny Merged Size: {tiny_merged_size}")
print(f"Number of actives in tiny merged dataset: {tiny_merged_actives}")
print(f"Number of actives in top 10%: {tiny_top_10_percent_actives}")
print(f"Number of actives in top 1%: {tiny_top_1_percent_actives}")
print(f"Tiny Merged EF: {tiny_merged_ef:.6f}")

tiny_merged_ef10 = calculate_ef10(tiny_merged)
print(f"\nTiny Merged EF10: {tiny_merged_ef10:.6f}")
print("=" * 50)

# Calculate EF metrics for tiny_merged_weighted dataset
tiny_merged_weighted_ef = calculate_ef(tiny_merged_weighted)
tiny_merged_weighted_size = len(tiny_merged_weighted)
tiny_merged_weighted_actives = tiny_merged_weighted['Active'].sum()
tiny_merged_weighted_ef1 = calculate_ef1(tiny_merged_weighted)

# Count actives in top 10% and top 1% for tiny_merged_weighted
tiny_weighted_top_10_percent_size = int(tiny_merged_weighted_size * 0.1)
tiny_weighted_top_1_percent_size = int(tiny_merged_weighted_size * 0.01)
tiny_weighted_top_10_percent_actives = tiny_merged_weighted.head(tiny_weighted_top_10_percent_size)['Active'].sum()
tiny_weighted_top_1_percent_actives = tiny_merged_weighted.head(tiny_weighted_top_1_percent_size)['Active'].sum()




print(f"\nTiny Merged Weighted Size: {tiny_merged_weighted_size}")
print(f"Number of actives in tiny merged weighted dataset: {tiny_merged_weighted_actives}")
print(f"Number of actives in top 10%: {tiny_weighted_top_10_percent_actives}")
print(f"Number of actives in top 1%: {tiny_weighted_top_1_percent_actives}")
print(f"Tiny Merged Weighted EF: {tiny_merged_weighted_ef:.6f}")
print(f"Tiny Merged Weighted EF1%: {tiny_merged_weighted_ef1:.6f}")

tiny_merged_weighted_ef10 = calculate_ef10(tiny_merged_weighted)
print(f"\nTiny Merged Weighted EF10: {tiny_merged_weighted_ef10:.6f}")

print("=" * 50)

globally_ef1 = calculate_ef1(merged_globally)
print(f"\nGlobally Merged EF1: {globally_ef1:.6f}")
globally_ef10 = calculate_ef10(merged_globally)
print(f"Globally Merged EF10: {globally_ef10:.6f}")
globally_size = len(merged_globally)
globally_actives = merged_globally['Active'].sum()
globally_top_1_percent_size = int(globally_size * 0.01)
globally_top_1_percent_actives = merged_globally.head(globally_top_1_percent_size)['Active'].sum()

print(f"Globally Merged Size: {globally_size}")
print(f"Number of actives in globally merged dataset: {globally_actives}")
print(f"Number of actives in top 1%: {globally_top_1_percent_actives}")


In [ ]:
nmdn_output = (
    nmdn_output
      .sort_values('pKd-Score', ascending=False)
      .reset_index(drop=True)           # ← drop old index, get 0…N−1
)

gnina_output = (
    gnina_output
      .sort_values('CNNaffinity', ascending=False)
      .reset_index(drop=True)
)

diffdock_output = (
    diffdock_output
      .sort_values('confidence_score', ascending=False)
      .reset_index(drop=True)
)

nmdn_output['rank'] = nmdn_output.index + 1
gnina_output['rank'] = gnina_output.index + 1
diffdock_output['rank'] = diffdock_output.index + 1 


import matplotlib.pyplot as plt
import seaborn as sns

def plot_rank_density_by_activity(
        df,
        rank_col: str = 'rank',
        active_col: str = 'Active',
        title: str = 'Rank density by activity',
        bw_adjust: float = 1.0
    ):
    """
    Overlaid KDEs of `rank_col`, clipped so the curve never extends
    beyond the min / max values present in the data.

    Parameters
    ----------
    df : pandas.DataFrame
    rank_col : str
        Column with 1-based rank integers.
    active_col : str
        Boolean or 0/1 column indicating actives.
    title : str
        Figure title.
    bw_adjust : float
        Passed straight to seaborn.kdeplot (bandwidth scale factor).
        Decrease (< 1) for sharper peaks; increase (> 1) for smoother.
    """
    rmin = df[rank_col].min()
    rmax = df[rank_col].max()

    plt.figure(figsize=(8, 4))
    sns.kdeplot(
        data=df[df[active_col]],
        x=rank_col,
        fill=True,
        color='green',
        label='Active',
        clip=(rmin, rmax),   # ← hard clip
        cut=0,               # ← stop kernel at the edges
        bw_adjust=bw_adjust,
    )
    sns.kdeplot(
        data=df[~df[active_col]],
        x=rank_col,
        fill=True,
        color='red',
        label='Inactive',
        clip=(rmin, rmax),
        cut=0,
        bw_adjust=bw_adjust,
    )

    plt.xlim(rmin, rmax)      # lock x-axis to observed range
    plt.xlabel('Rank')
    plt.ylabel('Density')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Example usage:
plot_rank_density_by_activity(nmdn_output, rank_col='rank', active_col='Active', title='NMDN Rank Density by Activity')
plot_rank_density_by_activity(gnina_output, rank_col='rank', active_col='Active', title='GNINA Rank Density by Activity')
plot_rank_density_by_activity(diffdock_output, rank_col='rank', active_col='Active', title='autodock Rank Density by Activity') 
plot_rank_density_by_activity(merged, rank_col='mean_rank', active_col='Active', title='Merged Rank Density by Activity')
plot_rank_density_by_activity(merged_globally, rank_col='mean_rank', active_col='Active', title='Globally Merged Rank Density by Activity')

import numpy as np
import matplotlib.pyplot as plt

def plot_rank_distribution(df, rank_col='rank', active_col='Active', title='Rank Distribution by Activity', bins=40):
    ranks = df[rank_col].to_numpy()
    labels = df[active_col].astype(bool).to_numpy()

    # split
    active_ranks   = ranks[labels]
    inactive_ranks = ranks[~labels]

    # plot
    plt.figure(figsize=(10, 6))
    plt.hist(inactive_ranks, bins=bins, alpha=0.6, density=True, label='Inactive')
    plt.hist(active_ranks,   bins=bins, alpha=0.6, density=True, label='Active')
    plt.title(title)
    plt.xlabel('Rank')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_rank_distribution(nmdn_output, rank_col='rank', active_col='Active', title='NMDN Rank Distribution by Activity')
plot_rank_distribution(gnina_output, rank_col='rank', active_col='Active', title='GNINA Rank Distribution by Activity')
plot_rank_distribution(diffdock_output, rank_col='rank', active_col='Active', title='autodock Rank Distribution by Activity')
plot_rank_distribution(merged, rank_col='mean_rank', active_col='Active', title='Merged Rank Distribution by Activity')
plot_rank_distribution(merged_globally, rank_col='mean_rank', active_col='Active', title='Globally Merged Rank Distribution by Activity')

In [ ]:
# Convert confidence_score to numeric, coercing errors to NaN
diffdock_output['confidence_score'] = pd.to_numeric(diffdock_output['confidence_score'], errors='coerce')

# Remove rows where confidence_score is NaN (was string or invalid)
diffdock_output = diffdock_output.dropna(subset=['confidence_score'])

print(f"After removing non-numeric confidence_score rows: {diffdock_output.shape}")
print(f"confidence_score data type: {diffdock_output['confidence_score'].dtype}")

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy import integrate

def calculate_bedroc(labels, scores, alpha=20.0, presorted=True):
    if not presorted:
        sorted_indices = np.argsort(scores)[::-1]
        labels = np.array(labels)[sorted_indices]
    else:
        labels = np.array(labels)

    n = len(labels)
    n_actives = np.sum(labels)
    if n_actives == 0 or n_actives == n:
        return np.nan

    # RIE calculation
    sum_term = 0.0
    for i, label in enumerate(labels, start=1):
        if label == 1:
            sum_term += np.exp(-alpha * i / n)

    cte = (n / n_actives) * (alpha / (1 - np.exp(-alpha)))
    rie = cte * sum_term

    # RIE min and max
    rie_min = cte * sum(np.exp(-alpha * i / n) for i in range(n - n_actives + 1, n + 1))
    rie_max = cte * sum(np.exp(-alpha * i / n) for i in range(1, n_actives + 1))

    # Normalize to BEDROC [0,1]
    bedroc = (rie - rie_min) / (rie_max - rie_min)
    return bedroc


    
def calculate_roce(labels, scores, alpha=20.0, presorted=True):
    """
    Calculate ROCE - assumes data is already sorted if presorted=True
    """
    if presorted:
        sorted_labels = np.array(labels)
    else:
        sorted_indices = np.argsort(scores)[::-1]
        sorted_labels = np.array(labels)[sorted_indices]
    
    n = len(labels)
    n_actives = np.sum(labels)
    
    if n_actives == 0 or n_actives == n:
        return np.nan
    
    # Calculate weighted sum
    weighted_sum = 0
    for i, label in enumerate(sorted_labels):
        if label == 1:
            weighted_sum += np.exp(-alpha * i / n)
    
    # Maximum possible sum (all actives at top)
    max_sum = sum(np.exp(-alpha * i / n) for i in range(n_actives))
    
    # Random expectation
    random_sum = n_actives * (1 - np.exp(-alpha)) / alpha
    
    # ROCE formula
    roce = (weighted_sum - random_sum) / (max_sum - random_sum)
    
    return roce

def calculate_ef_auc(labels, scores, presorted=True):
    """
    Calculate EF-AUC - assumes data is already sorted if presorted=True
    """
    if presorted:
        sorted_labels = np.array(labels)
    else:
        sorted_indices = np.argsort(scores)[::-1]
        sorted_labels = np.array(labels)[sorted_indices]
    
    n = len(labels)
    n_actives = np.sum(labels)
    
    if n_actives == 0:
        return np.nan
    
    # Calculate cumulative actives found
    cumulative_actives = np.cumsum(sorted_labels)
    
    # Calculate enrichment factors at each point
    fractions = np.arange(1, n + 1) / n
    ef_values = (cumulative_actives / np.arange(1, n + 1)) / (n_actives / n)
    
    # Integrate using trapezoidal rule
    ef_auc = integrate.trapz(ef_values, fractions)
    
    return ef_auc

def calculate_rie(labels, scores, alpha=20.0, presorted=True):
    """
    Calculate RIE - assumes data is already sorted if presorted=True
    """
    if presorted:
        sorted_labels = np.array(labels)
    else:
        sorted_indices = np.argsort(scores)[::-1]
        sorted_labels = np.array(labels)[sorted_indices]
    
    n = len(labels)
    n_actives = np.sum(labels)
    
    if n_actives == 0:
        return np.nan
    
    # Calculate RIE
    sum_term = 0
    for i, label in enumerate(sorted_labels):
        sum_term += label * np.exp(-alpha * i / n)
    
    # Expected value for random ranking
    expected_random = n_actives * (1 - np.exp(-alpha)) / alpha
    
    # Maximum possible value
    max_possible = sum(np.exp(-alpha * i / n) for i in range(min(n_actives, n)))
    
    # RIE formula
    rie = (sum_term - expected_random) / (max_possible - expected_random)
    
    return rie

def comprehensive_ranking_comparison(df, score_col, label_col='Active'):
    """
    Compare multiple ranking evaluation metrics
    Assumes data is already sorted correctly by score_col
    """
    # Clean data but don't re-sort
    df_clean = df.dropna(subset=[score_col, label_col]).copy()
    labels = df_clean[label_col].values
    scores = df_clean[score_col].values
    
    # Convert to numeric if needed
    scores = pd.to_numeric(scores, errors='coerce')
    df_clean = df_clean.dropna()
    labels = df_clean[label_col].values
    scores = df_clean[score_col].values
    
    results = {}
    
    try:
        # For ROC-AUC and PR-AUC, we still need the actual scores
        results['ROC-AUC'] = roc_auc_score(labels, scores)
        results['PR-AUC'] = average_precision_score(labels, scores)
        
        # For early recognition metrics, use the pre-sorted order
        results['BEDROC (α=20)'] = calculate_bedroc(labels, scores, alpha=20, presorted=True)
        results['BEDROC (α=50)'] = calculate_bedroc(labels, scores, alpha=50, presorted=True)
        results['ROCE (α=20)'] = calculate_roce(labels, scores, alpha=20, presorted=True)
        results['EF-AUC'] = calculate_ef_auc(labels, scores, presorted=True)
        results['RIE (α=20)'] = calculate_rie(labels, scores, alpha=20, presorted=True)
        
        # Traditional enrichment factors - use data as-is (already sorted)
        n = len(labels)
        n_actives = np.sum(labels)
        
        # NO SORTING - use the order as provided (already correctly sorted)
        sorted_labels = labels  # Data is already in correct order
        
        # EF at different fractions
        for frac in [0.01, 0.05, 0.10, 0.20]:
            k = max(1, int(n * frac))
            actives_in_top_k = np.sum(sorted_labels[:k])
            ef = (actives_in_top_k / k) / (n_actives / n) if n_actives > 0 else 0
            results[f'EF{int(frac*100)}%'] = ef
            
    except Exception as e:
        print(f"Error calculating metrics: {e}")
        return {}
    
    return results

# Compare all three methods with comprehensive metrics
print("=== COMPREHENSIVE RANKING COMPARISON ===\n")

methods = [
    ('autodock', diffdock_output, 'confidence_score'),
    ('NMDN', nmdn_output, 'pKd-Score'),
    ('GNINA', gnina_output, 'CNNaffinity'),
    ('Merged', merged, 'mean_rank'),
    ('tiny Merged', tiny_merged, 'mean_rank'),
    ('tiny Merged Weighted', tiny_merged_weighted, 'mean_rank'),
    ('Globally Merged', merged_globally, 'mean_rank'),
]

comparison_results = {}

for method_name, df, score_col in methods:
    print(f"--- {method_name} ---")
    results = comprehensive_ranking_comparison(df, score_col)
    comparison_results[method_name] = results
    
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")
    print()

# Create comparison table
comparison_df = pd.DataFrame(comparison_results).T
print("=== COMPARISON TABLE ===")
print(comparison_df.round(4))

# Save results
# comparison_df.to_csv('comprehensive_ranking_comparison.csv')
# print("\nResults saved to 'comprehensive_ranking_comparison.csv'")

In [ ]:
total_actives = nmdn_output['Active'].sum()
total_drugs = nmdn_output.shape[0]
ratio = total_actives / total_drugs if total_drugs > 0 else 0

def calculate_efficacy_ratio(df, original_active_rate):
    # Calculate the number of active compounds in the current dataframe
    if len(df) == 0:
        return 0
    num_actives = df['Active'].sum()
    # Calculate the total number of compounds in the current dataframe
    total_compounds = df.shape[0]
    active_rate = num_actives / total_compounds 
    # Calculate the efficacy ratio
    efficacy_ratio = active_rate / original_active_rate 
    return efficacy_ratio

efficacy_ratio_merged = calculate_efficacy_ratio(merged, ratio)
efficacy_ratio_tiny_merged = calculate_efficacy_ratio(tiny_merged, ratio)
efficacy_ratio_tiny_merged_weighted = calculate_efficacy_ratio(tiny_merged_weighted, ratio)
efficacy_ratio_globally_merged = calculate_efficacy_ratio(merged_globally, ratio)

actives_merged = merged['Active'].sum()
actives_percentage_merged = (actives_merged / total_actives * 100) if total_actives > 0 else 0
actives_tiny_merged = tiny_merged['Active'].sum()
actives_percentage_tiny_merged = (actives_tiny_merged / total_actives * 100) if total_actives > 0 else 0
actives_tiny_merged_weighted = tiny_merged_weighted['Active'].sum()
actives_percentage_tiny_merged_weighted = (actives_tiny_merged_weighted / total_actives * 100) if total_actives > 0 else 0
actives_globally_merged = merged_globally['Active'].sum()
actives_percentage_globally_merged = (actives_globally_merged / total_actives * 100) if total_actives > 0 else 0

print(f"Actives originally {total_actives}")

print("=" * 50)
print(f"Efficacy ratio (Filtered Calibrated 221): {efficacy_ratio_merged}")
print(f"Active compounds (Filtered Calibrated 221): {actives_merged}")
print(f"Active compounds (Filtered Calibrated 221) percentage: {actives_percentage_merged:.2f}%")
print("=" * 50)
print(f"Efficacy ratio (filtered_900_0.6): {efficacy_ratio_tiny_merged}")
print(f"Active compounds (filtered_900_0.6): {actives_tiny_merged}")
print(f"Active compounds (filtered_900_0.6) percentage: {actives_percentage_tiny_merged:.2f}%")
print("=" * 50)
print(f"Efficacy ratio (calibrated_221_unfiltered): {efficacy_ratio_tiny_merged_weighted}")
print(f"Active compounds (calibrated_221_unfiltered): {actives_tiny_merged_weighted}")
print(f"Active compounds (calibrated_221_unfiltered) percentage: {actives_percentage_tiny_merged_weighted:.2f}%")
print("=" * 50)
print(f"Efficacy ratio (Globally Merged): {efficacy_ratio_globally_merged}")
print(f"Active compounds (Globally Merged): {actives_globally_merged}")
print(f"Active compounds (Globally Merged) percentage: {actives_percentage_globally_merged:.2f}%")





